# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR\(^2\) mass spectrometry dataset using the `mlcroissant` library. All interactions use `@id` fields for record sets, fields, and columns to ensure reproducibility and clarity.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This section imports necessary libraries and fetches information from the provided Croissant URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Name :", metadata.name)
print("Description :", metadata.description)

## 2. Data Overview
Review available record sets, fields, and their `@id` values using the `mlcroissant` dataset interface. This will help you identify the structure and contents of the dataset for further processing.

In [ ]:
# Show available record sets and their fields with @id

# Helper function to print all available record sets, fields and columns
def print_dataset_structure(ds):
    record_sets = ds.metadata.record_sets
    if not record_sets:
        print("No record sets defined in the dataset.")
        return
    print(f"Found {len(record_sets)} record set(s):\n")
    for rs in record_sets:
        print(f"- RecordSet: {rs.name} (@id={rs.id})")
        print(f"  Description: {rs.description}")
        if rs.fields:
            print(f"  Fields:")
            for field in rs.fields:
                print(f"    - Field name: {field.name}, @id: {field.id}, dataType: {getattr(field, 'data_type', None)}")
                if hasattr(field, 'columns') and field.columns:
                    print(f"      Columns:")
                    for col in field.columns:
                        print(f"        - Column name: {col.name}, @id: {col.id}, dataType: {getattr(col, 'data_type', None)}")
        print("")
        
print_dataset_structure(dataset)


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Always use each record set's and field's `@id`. Typically mass spec datasets will have one main data table.


In [ ]:
# List available record set @ids
record_sets = [rs.id for rs in dataset.metadata.record_sets]
print("Record set @ids:", record_sets)

dataframes = {}
for record_set_id in record_sets:
    print(f"Loading record set {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
    else:
        print(f"No records found for record set {record_set_id}.")

# For example, pick the first record set (if available)
if record_sets:
    main_record_set_id = record_sets[0]
    if main_record_set_id in dataframes:
        print(f"Columns in record set {main_record_set_id}:")
        print(dataframes[main_record_set_id].columns.tolist())
        display(dataframes[main_record_set_id].head())
    else:
        print("No data available for main record set.")
else:
    main_record_set_id = None


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records based on a numeric field, normalizing, and grouping. Replace `<numeric_field_id>` with an actual field @id identified above.


In [ ]:
if dataframes.get(main_record_set_id) is not None:
    df = dataframes[main_record_set_id].copy()

    # Find numeric columns that can be analyzed
    numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
    print("Numeric fields available:", numeric_fields)
    numeric_field = numeric_fields[0] if numeric_fields else None

    if numeric_field:
        print(f"Analyzing numeric field: {numeric_field}")
        threshold = df[numeric_field].mean() if df[numeric_field].mean() > 0 else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize numeric field
        field_norm = f"{numeric_field}_normalized"
        filtered_df[field_norm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, field_norm]].head())

        # Try grouping by another field (e.g., first non-numeric field)
        non_numeric_fields = [c for c in df.columns if c not in numeric_fields]
        group_field = non_numeric_fields[0] if non_numeric_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (mean of {numeric_field}):")
            display(grouped_df.head())
    else:
        print("No numeric fields identified for EDA.")
else:
    print("No dataframe available for EDA.")


## 5. Visualization
Visualize the distribution of a major numeric field, e.g., as a histogram (and possibly by group, if meaningful).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes.get(main_record_set_id) is not None and numeric_field:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} in record set {main_record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Optionally, show grouped boxplot if a group_field is available
    if group_field and df[group_field].nunique() < 10:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"Boxplot of {numeric_field} grouped by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()


## 6. Conclusion
This notebook illustrated loading, exploration, and basic processing of the FAIR\(^2\) mass spectrometry dataset using `mlcroissant`, with all entities referenced by their `@id` fields. Further biological, comparative, or biomarker analysis can be enabled by leveraging the structured metadata and the main data table.